# Set Up

In [ ]:
import pandas as pd
import numpy as np
import json
import copy
import time
import os
import gc
import random
import joblib
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import torch._dynamo
torch._dynamo.config.capture_scalar_outputs = True

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate

In [ ]:
import sys
from pathlib import Path

quantnets_path = Path("/scratch/bng/cartbind/code/MIND_models/QuantNets")
if str(quantnets_path) not in sys.path:
    sys.path.append(str(quantnets_path))

from gnn.architectures_inject import GATv2ConvNet as GATv2ConvNet_Inject
from gnn.architectures_normal import GATv2ConvNet as GATv2ConvNet_Normal

from gnn_training_pipeline.metrics import calc_r2_corr

# Configuration

In [ ]:
# --- Determinism ---
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

torch.set_float32_matmul_precision('high')
USE_AMP = False
USE_PRUNER = False

# --- Configuration ---
base_dir = Path('/scratch/bng/cartbind/code/MIND_models')
data_dir = Path('/scratch/bng/cartbind/data/UKB_new_data/combined_data_no_outliers')
splits_dir = base_dir / 'scaling_law_splits'
region_dir = base_dir / 'region_names'

gnn_dir = base_dir / 'models_gnn_dnanexus'
results_dir = gnn_dir / 'gnn_scaling_law_results'
predictions_dir = gnn_dir / 'gnn_predictions_scaling_law'
plots_dir = gnn_dir / 'gnn_scaling_law_plots'
params_dir = gnn_dir / 'gnn_best_params_scaling_law'
weights_dir = gnn_dir / 'gnn_weights_scaling_law'
preprocessors_dir = gnn_dir / 'gnn_preprocessors_scaling_law'

for d in [results_dir, predictions_dir, plots_dir, params_dir, weights_dir, preprocessors_dir]:
    d.mkdir(parents=True, exist_ok=True)

targets = {
    'GF': ('GF', 'p20016_i2'),
    # 'PAL': ('PAL', 'p20197_i2'),
    # 'DSST': ('DSST', 'p23324_i2'),
    # 'TMT': ('TMT', 'p6350_i2'),
}

data_configs = {
    # 'FC25': (region_dir / 'FC25_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    'FC100': (region_dir / 'FC100_regions.txt', ['p31', 'p21003_i2', 'p54_i2', 'p25741_i2']),
    # 'MIND': (region_dir / 'MIND_regions.txt', ['p31', 'p21003_i2', 'p54_i2']),
}

# sample_sizes = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 'all']
sample_sizes = ['all']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

MAX_EPOCHS = 200
MIN_RESOURCE = 30
WARMUP_EPOCHS = 10
EARLY_STOP_PATIENCE = 15
N_TRIALS = 100
BATCH_SIZE = 512

In [ ]:
# Helper from process_graph_pipeline/csv_to_graph.py
def parse_region_map(region_file_path):
    with open(region_file_path, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    raw_pairs = []
    for line in lines:
        if line.startswith("FC"):
            inner = line[line.find("(")+1:line.find(")")]
            src, dst = inner.split("-")
        else:
            src, dst = line.split("-")
        raw_pairs.append((src, dst))

    unique_nodes = []
    for src, dst in raw_pairs:
        if src not in unique_nodes: unique_nodes.append(src)
        if dst not in unique_nodes: unique_nodes.append(dst)
        
    node_to_idx = {name: idx for idx, name in enumerate(unique_nodes)}
    edge_list = [(node_to_idx[src], node_to_idx[dst]) for src, dst in raw_pairs]
    
    return len(unique_nodes), edge_list, lines


class BrainGraphDataset(torch.utils.data.Dataset):
    def __init__(self, X_edge_features_scaled, X_edge_features_raw, demographics_features, 
                 y_targets, eids, num_nodes, mapping_edges, sparsity_percent):
        """
        X_edge_features_scaled: Scaled flattened edges for node connections
        X_edge_features_raw: Unscaled flattened edges used strictly for top-k sparsification 
        demographics_features: NumPy array of patient traits (num_samples, num_demo_feats) 
        y_targets: target regression array
        eids: standard subjects 
        sparsity_percent: float from 0.0 to 1.0 (1.0 = keep all edges)
        """
        # Format eids exactly once here
        self.eids = eids.to_numpy() if hasattr(eids, 'to_numpy') else np.array(eids)
        
        num_total_edges = len(mapping_edges)
        k_edges_to_keep = max(1, int(np.floor(sparsity_percent * num_total_edges)))

        src_global = np.array([e[0] for e in mapping_edges])
        dst_global = np.array([e[1] for e in mapping_edges])

        # Pre-reshape labels and demographics globally
        X_demo = torch.tensor(demographics_features, dtype=torch.float).unsqueeze(1) # Shape: (N, 1, D)
        targets = torch.tensor(y_targets, dtype=torch.float).unsqueeze(1)            # Shape: (N, 1)
        
        # Create the node list exactly once
        x_node = torch.arange(num_nodes, dtype=torch.long)
        
        # Pre-build the full list of Data objects
        self.data_list = []
        
        # Partitions the entire 2D raw feature array at once
        if k_edges_to_keep < num_total_edges:
            all_top_idx = np.argpartition(
                np.abs(X_edge_features_raw), -k_edges_to_keep, axis=1
            )[:, -k_edges_to_keep:]
        else:
            all_top_idx = None
            
        for idx in range(len(self.eids)):
            w_u_scaled = X_edge_features_scaled[idx]
            
            # Retrieve pre-computed row mask
            if all_top_idx is not None:
                top_idx = all_top_idx[idx]
            else:
                top_idx = np.arange(num_total_edges)

            w_selected = w_u_scaled[top_idx]
            src_u = src_global[top_idx]
            dst_u = dst_global[top_idx]
            
            # Stack once, then repeat to create undirected edges
            # Shape becomes (2, 2 * k_edges)
            edge_index_np = np.hstack([
                np.vstack([src_u, dst_u]), 
                np.vstack([dst_u, src_u])
            ])
            edge_index = torch.tensor(edge_index_np, dtype=torch.long)
            
            w = np.tile(w_selected, 2)
            edge_attr = torch.tensor(w, dtype=torch.float).unsqueeze(-1)
            
            graph_data = Data(
                x=x_node, 
                edge_index=edge_index, 
                edge_attr=edge_attr,
                demographics=X_demo[idx],
                y=targets[idx],
                eid=self.eids[idx]
            )
            self.data_list.append(graph_data)
        
    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

In [ ]:
for data_name, (regions_file, demographic_cols) in data_configs.items():
    num_nodes, mapping_edges, brain_cols = parse_region_map(regions_file)
    print(f"{data_name}: {num_nodes} nodes")

# Optuna and analysis functions

In [ ]:
def train_gnn(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        with autocast(device_type='cuda', enabled=USE_AMP):
            out = model(data)
            loss = criterion(out, data.y.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)

@torch.no_grad()
def eval_gnn(model, loader, criterion):
    model.eval()
    total_loss = 0
    preds, actuals, eids = [], [], []
    for data in loader:
        data = data.to(device)
        with autocast(device_type='cuda', enabled=USE_AMP):
            out = model(data)
            loss = criterion(out, data.y.view(-1))
        total_loss += loss.item() * data.num_graphs
        
        preds.extend(out.cpu().float().view(-1).numpy())
        actuals.extend(data.y.cpu().view(-1).numpy())
        if hasattr(data.eid, 'cpu'):
            eids.extend(data.eid.cpu().numpy())
        else:
            eids.extend(data.eid)
    return (total_loss / len(loader.dataset)), np.array(preds), np.array(actuals), np.array(eids)


def logging_callback(study, trial):
    # study.best_trial might raise ValueError if no trials have successfully completed yet
    try:
        best_trial = study.best_trial
        best_test_r2_corr = best_trial.user_attrs.get('test_r2_corr', None)
        
        if best_test_r2_corr is not None:
            print(f"Trial {trial.number} finished. Best Trial so far: {best_trial.number} | Best Val Loss: {best_trial.value:.4f} | Best Test R² Corr: {best_test_r2_corr:.4f}")
    except ValueError:
        pass

In [ ]:
def get_optimizer_params(model, weight_decay):
    # PyG and PyTorch use 'bias' for biases, and typical norms have 'norm' in their layer names.
    no_decay = ['bias', 'norm'] 
    
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n.lower() for nd in no_decay)], 
            "weight_decay": weight_decay
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n.lower() for nd in no_decay)], 
            "weight_decay": 0.0
        },
    ]
    return optimizer_grouped_parameters

def evaluate_loader(model, loader, criterion, y_preprocessor):
    _, preds_scaled, actuals_scaled, eids = eval_gnn(model, loader, criterion)
    preds = y_preprocessor.inverse_transform(preds_scaled.reshape(-1, 1)).flatten()
    actuals = y_preprocessor.inverse_transform(actuals_scaled.reshape(-1, 1)).flatten()
    return actuals, preds, eids

In [ ]:
def objective(trial, 
              X_brain_train_scaled, X_brain_train_raw, X_demo_train, y_train, eids_train, 
              X_brain_val_scaled, X_brain_val_raw, X_demo_val, y_val, eids_val, 
              X_brain_test_scaled, X_brain_test_raw, X_demo_test, y_test, eids_test,
              num_nodes, mapping_edges, y_preprocessor, trial_weights, trial_artifacts):
    
    start_time = time.time()
    
    # Suggest hyperparameters (including architecture)
    # architecture = trial.suggest_categorical('architecture', ['normal', 'inject'])
    architecture = trial.suggest_categorical('architecture', ['inject'])

    hidden_channels_pow = trial.suggest_int('hidden_channels', 5, 7)
    hidden_channels = 2 ** hidden_channels_pow
    embedding_dim_pow = trial.suggest_int('embedding_dim', 5, hidden_channels_pow)
    embedding_dim = 2 ** embedding_dim_pow
    gnn_num_layers = trial.suggest_int('gnn_num_layers', 2, 4)

    # Sample classifier powers to ensure strictly decreasing dimensions
    classifier_num_layers = trial.suggest_int('classifier_num_layers', 1, 3)
    classifier_hidden_dims = []
    current_upper_bound = hidden_channels_pow - 1
    
    for i in range(classifier_num_layers):
        # The lowest possible power we can pick to still have room for the remaining layers
        min_possible = classifier_num_layers - i 
        lower_bound = max(1, min_possible)
        power = trial.suggest_int(f"classifier_dim_exp_l{i}", lower_bound, current_upper_bound)
        classifier_hidden_dims.append(2 ** power)
        current_upper_bound = power - 1

    hidden_heads_pow = trial.suggest_int('hidden_heads', 1, min(4, max(0, hidden_channels_pow - 2)))
    hidden_heads = 2 ** hidden_heads_pow
    demo_embed_dim_pow = trial.suggest_int('demo_embed_dim', 1, min(4, hidden_channels_pow))
    demo_embed_dim = 2 ** demo_embed_dim_pow

    edge_dropout = trial.suggest_float('edge_dropout', 0.0, 0.7, step=0.05)
    node_dropout = trial.suggest_float('node_dropout', 0.0, 0.7, step=0.05)
    classifier_dropout = trial.suggest_float('classifier_dropout', 0.0, 0.7, step=0.05)
    # concat = trial.suggest_categorical('concat', [True, False])
    concat = trial.suggest_categorical('concat', [False])
    # include_jk = trial.suggest_categorical('include_jk', [True, False])
    include_jk = trial.suggest_categorical('include_jk', [False])
    
    lr = trial.suggest_float('lr', 1e-3, 3e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 5e-1, log=True)
    # sparsity_percent = trial.suggest_float('sparsity_percent', 0.3, 1.0, step=0.05) 
    sparsity_percent = trial.suggest_float('sparsity_percent', 1.0, 1.0, step=0.05) 
    
    try:
        # Initialize datasets with both scaled and raw arrays
        train_dataset = BrainGraphDataset(X_brain_train_scaled, X_brain_train_raw, X_demo_train, y_train, eids_train, num_nodes, mapping_edges, sparsity_percent)
        val_dataset = BrainGraphDataset(X_brain_val_scaled, X_brain_val_raw, X_demo_val, y_val, eids_val, num_nodes, mapping_edges, sparsity_percent)
        test_dataset = BrainGraphDataset(X_brain_test_scaled, X_brain_test_raw, X_demo_test, y_test, eids_test, num_nodes, mapping_edges, sparsity_percent)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

        num_demo_features = X_demo_train.shape[1] 
        
        # Select the model class based on Optuna's suggestion
        ModelClass = GATv2ConvNet_Inject if architecture == 'inject' else GATv2ConvNet_Normal

        # Initialize new dynamic model
        model = ModelClass(
            out_dim=1, 
            in_channels=num_nodes,          
            gnn_num_layers=gnn_num_layers, 
            classifier_hidden_dims=classifier_hidden_dims,
            hidden_channels=hidden_channels, 
            embedding_dim=embedding_dim,
            demo_dim=num_demo_features,
            edge_dropout_rate=edge_dropout,
            node_dropout_rate=node_dropout,
            hidden_heads=hidden_heads,
            demo_embed_dim=demo_embed_dim,
            include_demo=True,
            classifier_dropout_rate=classifier_dropout,
            concat=concat,
            include_jk=include_jk
        ).to(device)

        model = torch.compile(model, dynamic=True)
        
        param_groups = get_optimizer_params(model, weight_decay)
        optimizer = torch.optim.AdamW(param_groups, lr=lr)
        criterion = nn.MSELoss()

        warmup_scheduler = LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS)
        decay_epochs = MAX_EPOCHS - WARMUP_EPOCHS
        cosine_scheduler = CosineAnnealingLR(optimizer, T_max=decay_epochs)
        scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS])

        best_val_loss = float('inf')
        best_epoch = 0
        early_stop_patience = EARLY_STOP_PATIENCE
        no_improve = 0
        best_model_state = None
        
        scaler = GradScaler(enabled=USE_AMP)

        train_losses = []
        val_losses = []

        with tqdm(range(MAX_EPOCHS), desc="Training Epochs", leave=False) as pbar:
            for epoch in pbar:
                train_loss = train_gnn(model, train_loader, optimizer, criterion, scaler)

                if np.isnan(train_loss) or np.isinf(train_loss):
                    pbar.close()
                    raise optuna.exceptions.TrialPruned()

                scheduler.step()
                val_loss, _, _, _ = eval_gnn(model, val_loader, criterion)

                train_losses.append(train_loss)
                val_losses.append(val_loss)

                if np.isnan(val_loss) or np.isinf(val_loss):
                    pbar.close()
                    raise optuna.exceptions.TrialPruned()
                
                trial.report(val_loss, epoch)
                if USE_PRUNER and trial.should_prune():
                    pbar.close()
                    raise optuna.exceptions.TrialPruned()
                    
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_epoch = epoch + 1 
                    no_improve = 0
                    best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                else:
                    no_improve += 1
                    if no_improve >= early_stop_patience:
                        pbar.close()
                        break
                    
        # --- Post-Training Evaluation on Best State ---
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            
            train_actuals, train_preds, _ = evaluate_loader(model, train_loader, criterion, y_preprocessor)
            val_actuals, val_preds, _ = evaluate_loader(model, val_loader, criterion, y_preprocessor)
            test_actuals, test_preds, test_eids = evaluate_loader(model, test_loader, criterion, y_preprocessor)

            try:
                current_global_best = trial.study.best_value
            except ValueError:
                current_global_best = float('inf')

            if best_val_loss < current_global_best:
                trial_weights.clear()
                trial_weights['weights'] = best_model_state

                trial_artifacts.clear()
                trial_artifacts['train_losses'] = train_losses
                trial_artifacts['val_losses'] = val_losses
                trial_artifacts['test_actuals'] = test_actuals
                trial_artifacts['test_preds'] = test_preds
                trial_artifacts['test_eids'] = test_eids

            # Save lightweight scalars to Optuna for your visual plots
            trial.set_user_attr('train_r2', r2_score(train_actuals, train_preds))
            trial.set_user_attr('train_r2_corr', calc_r2_corr(train_actuals, train_preds))
            
            trial.set_user_attr('val_r2', r2_score(val_actuals, val_preds))
            trial.set_user_attr('val_r2_corr', calc_r2_corr(val_actuals, val_preds))
            
            trial.set_user_attr('test_r2', r2_score(test_actuals, test_preds))
            trial.set_user_attr('test_r2_corr', calc_r2_corr(test_actuals, test_preds))

            trial.set_user_attr('best_epoch', best_epoch)
            trial.set_user_attr('time_elapsed_mins', (time.time() - start_time) / 60.0)
            
        return best_val_loss
    
    finally:
        if 'model' in locals(): del model
        if 'optimizer' in locals(): del optimizer
        if 'scheduler' in locals(): del scheduler
        if 'scaler' in locals(): del scaler
        if 'train_loader' in locals(): del train_loader
        if 'val_loader' in locals(): del val_loader
        if 'test_loader' in locals(): del test_loader
        if 'train_dataset' in locals(): del train_dataset
        if 'val_dataset' in locals(): del val_dataset
        if 'test_dataset' in locals(): del test_dataset
            
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        torch._dynamo.reset()

In [ ]:
def gnn_analysis(df, data_name, target_name, sample_size):
    # Grab the relevant properties for the mapping from data_configs
    regions_file = data_configs[data_name][0]
    demographic_cols = data_configs[data_name][1]

    # Parse structural nodes & edges
    num_nodes, mapping_edges, brain_cols = parse_region_map(regions_file)
    
    eids = df.index if 'eid' not in df.columns else df['eid']
    y = df[targets[target_name][1]]
    
    # Figure out categorical vs numerical bounds for the demographic pipeline
    cat_vars = [c for c in ['sex', 'assessment_centre', 'p31', 'p54_i2'] if c in demographic_cols]
    num_vars = [c for c in demographic_cols if c not in cat_vars]

    target_predictions_dir = predictions_dir / target_name
    target_predictions_dir.mkdir(parents=True, exist_ok=True)
    preds_path = target_predictions_dir / f'GNN_preds_{data_name}_{target_name}_{sample_size}.csv'

    target_params_dir = params_dir / target_name
    target_params_dir.mkdir(parents=True, exist_ok=True)
    params_path = target_params_dir / f'GNN_best_params_{data_name}_{target_name}_{sample_size}.csv'

    # Ensure ALL expected graph edges are present as columns
    missing_brain_cols = [c for c in brain_cols if c not in df.columns]
    if missing_brain_cols:
        raise ValueError(f"Missing {len(missing_brain_cols)} expected brain edge columns in dataset (e.g., {missing_brain_cols[:3]})")
    X_brain_raw = df[brain_cols]
    # Raise error if any patient has missing edge values
    if X_brain_raw.isna().any().any():
        raise ValueError("Missing data (NaNs) detected in the brain edge features. Patients must have 100% complete FC matrices.")
    X_demo_raw = df[demographic_cols]    
    # Raise error if any patient has missing demographic values
    if X_demo_raw.isna().any().any():
        raise ValueError("Missing data (NaNs) detected in the demographic features.")        
    # Raise error if any patient is missing the target label
    if y.isna().any():
        raise ValueError(f"Missing data (NaNs) detected in the target variable '{targets[target_name][1]}'.")
    
    run_start_time = time.time()

    # --- Split Data: 70% Train, 15% Val, 15% Test ---
    # First split off 30% for val and test combined
    train_idx, temp_idx = train_test_split(
        np.arange(len(eids)), test_size=0.30, random_state=seed
    )
    # Then split the 30% temp array into roughly equal halves for 15% val and 15% test
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=seed
    )

    print(f"Dataset split - Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

    # Scaling
    demo_preprocessor = ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_vars),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_vars)
    ])
    edge_preprocessor = StandardScaler()
    y_preprocessor = StandardScaler()

    # Fit & Transform ONLY on Train
    X_brain_train = edge_preprocessor.fit_transform(X_brain_raw.iloc[train_idx])
    X_demo_train = demo_preprocessor.fit_transform(X_demo_raw.iloc[train_idx])
    y_train = y_preprocessor.fit_transform(y.iloc[train_idx].values.reshape(-1, 1)).flatten()
    eids_train = eids[train_idx]

    # Transform Val
    X_brain_val = edge_preprocessor.transform(X_brain_raw.iloc[val_idx])
    X_demo_val = demo_preprocessor.transform(X_demo_raw.iloc[val_idx])
    y_val = y_preprocessor.transform(y.iloc[val_idx].values.reshape(-1, 1)).flatten()
    eids_val = eids[val_idx]

    # Transform Test
    X_brain_test = edge_preprocessor.transform(X_brain_raw.iloc[test_idx])
    X_demo_test = demo_preprocessor.transform(X_demo_raw.iloc[test_idx])
    y_test_scaled = y_preprocessor.transform(y.iloc[test_idx].values.reshape(-1, 1)).flatten()
    eids_test = eids[test_idx]
    
    # Optuna Optimization
    trial_weights = {}
    trial_artifacts = {}

    objective_func = lambda trial: objective(
        trial, 
        X_brain_train, X_brain_raw.iloc[train_idx].values, X_demo_train, y_train, eids_train, 
        X_brain_val, X_brain_raw.iloc[val_idx].values, X_demo_val, y_val, eids_val, 
        X_brain_test, X_brain_raw.iloc[test_idx].values, X_demo_test, y_test_scaled, eids_test,
        num_nodes, mapping_edges, y_preprocessor, trial_weights, trial_artifacts
    )
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    pruner = optuna.pruners.HyperbandPruner(min_resource=MIN_RESOURCE, max_resource=MAX_EPOCHS, reduction_factor=3) if USE_PRUNER else None
    db_path = f"sqlite:///{params_dir}/study_{target_name}_{data_name}_{sample_size}.db"
    study = optuna.create_study(
        study_name=f"run_nocv",
        direction='minimize', 
        sampler=optuna.samplers.TPESampler(seed=seed), 
        pruner=pruner,
        storage=db_path,
        load_if_exists=True
    )

    study.optimize(
        objective_func, 
        n_trials=N_TRIALS, 
        show_progress_bar=True, 
        callbacks=[logging_callback]
    )

    # Output graphs cleanly
    plot_dir = plots_dir / target_name / data_name / f'n_{sample_size}'
    plot_dir.mkdir(parents=True, exist_ok=True)

    fig_history = plot_optimization_history(study)
    
    # Inject custom user_attrs into hovertemplate
    for trace in fig_history.data:
        if trace.mode == 'markers':
            custom_attrs = []
            for trial_idx in trace.x:
                t = study.trials[int(trial_idx)]
                
                tr_r2 = t.user_attrs.get('train_r2', 'N/A')
                va_r2 = t.user_attrs.get('val_r2', 'N/A')
                te_r2 = t.user_attrs.get('test_r2', 'N/A')

                tr_r2_corr = t.user_attrs.get('train_r2_corr', 'N/A')
                va_r2_corr = t.user_attrs.get('val_r2_corr', 'N/A')
                te_r2_corr = t.user_attrs.get('test_r2_corr', 'N/A')

                ep = t.user_attrs.get('best_epoch', 'N/A')
                tm = t.user_attrs.get('time_elapsed_mins', 'N/A')

                fmt = lambda v: f"{v:.3f}" if isinstance(v, float) else str(v)

                params_str = "".join([f"<br>  {k}: {v}" for k, v in t.params.items()])

                custom_attrs.append(
                    f"<br>Epochs: {ep}<br>Time: {fmt(tm)}m"
                    f"<br>Train R²: {fmt(tr_r2)} | R² Corr: {fmt(tr_r2_corr)}"
                    f"<br>Val R²: {fmt(va_r2)} | R² Corr: {fmt(va_r2_corr)}"
                    f"<br>Test R²: {fmt(te_r2)} | R² Corr: {fmt(te_r2_corr)}"
                    f"<br><b>Params:</b>{params_str}"
                )
            trace.customdata = custom_attrs
            
            if trace.hovertemplate is None:
                trace.hovertemplate = "Trial: %{x}<br>Objective Value: %{y}%{customdata}"
            else:
                trace.hovertemplate += "%{customdata}"

    fig_history.write_image(str(plot_dir / 'optimization_history.png'))
    fig_history.write_html(str(plot_dir / 'optimization_history.html'))

    # Grab test metrics directly from Optuna best trial
    best_trial = study.best_trial

    plot_params = [p for p in best_trial.params.keys() if not p.startswith("classifier_dim_exp")]
    fig_parallel = plot_parallel_coordinate(study, params=plot_params)
    fig_parallel.write_image(str(plot_dir / 'parallel_coordinate.png'))
    fig_parallel.write_html(str(plot_dir / 'parallel_coordinate.html'))
    fig_importances = plot_param_importances(study)
    fig_importances.write_image(str(plot_dir / 'param_importances.png'))
    fig_importances.write_html(str(plot_dir / 'param_importances.html'))

    # Save Best Model Weights
    target_weights_dir = weights_dir / target_name
    target_weights_dir.mkdir(parents=True, exist_ok=True)
    weights_path = target_weights_dir / f'GNN_weights_{data_name}_{target_name}_{sample_size}.pt'
    torch.save(trial_weights['weights'], weights_path)

    # Save Preprocessors
    target_preprocessors_dir = preprocessors_dir / target_name
    target_preprocessors_dir.mkdir(parents=True, exist_ok=True)
    preprocessors_path = target_preprocessors_dir / f'preprocessors_{data_name}_{target_name}_{sample_size}.pkl'
    joblib.dump({
        'demo': demo_preprocessor,
        'edge': edge_preprocessor,
        'y': y_preprocessor
    }, preprocessors_path)

    # Save the losses
    losses_path = target_weights_dir / f'GNN_losses_{data_name}_{target_name}_{sample_size}.csv'
    pd.DataFrame({
        'train_loss': trial_artifacts['train_losses'],
        'val_loss': trial_artifacts['val_losses']
    }).to_csv(losses_path, index_label='epoch')

    best_actuals = trial_artifacts['test_actuals']
    best_preds = trial_artifacts['test_preds']
    best_eids = trial_artifacts['test_eids']  

    best_mae = mean_absolute_error(best_actuals, best_preds)
    best_rmse = np.sqrt(mean_squared_error(best_actuals, best_preds))

    pruned_trials_count = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])

    # Save predictions to CSV cleanly
    run_df = pd.DataFrame({'eid': best_eids, 'actual': best_actuals, 'predicted': best_preds})
    run_df.to_csv(preds_path, mode='w', index=False)

    # Save Best Parameters 
    params_dict = {
        'best_trial_number': best_trial.number,
        'best_epoch': best_trial.user_attrs.get('best_epoch'),
        'time_elapsed_mins': best_trial.user_attrs.get('time_elapsed_mins')
    }
    params_dict.update(best_trial.params)
    params_df = pd.DataFrame([params_dict])
    params_df.to_csv(params_path, mode='w', index=False)

    elapsed_mins = (time.time() - run_start_time) / 60.0
    run_summary = f'  Run Completed ({elapsed_mins:.1f}m) • Pruned: {pruned_trials_count}/{N_TRIALS} • MAE={best_mae:.3f} • RMSE={best_rmse:.3f} • Test R²={best_trial.user_attrs["test_r2"]:.3f}'        
    print(run_summary)

    del study, objective_func
    del X_brain_train, X_demo_train, y_train
    del X_brain_val, X_demo_val, y_val
    del X_brain_test, X_demo_test, y_test_scaled
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'test_mae': best_mae, 
        'test_rmse': best_rmse,
        
        'train_r2': best_trial.user_attrs['train_r2'], 
        'train_r2_corr': best_trial.user_attrs['train_r2_corr'], 
        
        'val_r2': best_trial.user_attrs['val_r2'], 
        'val_r2_corr': best_trial.user_attrs['val_r2_corr'], 
        
        'test_r2': best_trial.user_attrs['test_r2'], 
        'test_r2_corr': best_trial.user_attrs['test_r2_corr'], 
    }

# Scaling Law Training Loop

In [ ]:
for target_name, (test_key, score_col) in targets.items():
    print(f'\n{"="*60}\nTARGET: {target_name}\n{"="*60}')
    data_file = data_dir / f'combined_data_{test_key}_no_outliers.csv'
    target_splits_dir = splits_dir / target_name
    df_full = pd.read_csv(data_file)
    if 'eid' in df_full.columns:
        df_full.set_index('eid', inplace=True)
        
    for data_name in data_configs.keys():
        print(f'\n--- {target_name} vs. {data_name} ---')
        for sample_size in sample_sizes:
            eid_file = (target_splits_dir / f'{target_name}_all_eids.txt') if sample_size == 'all' else (target_splits_dir / f'{target_name}_eids_{sample_size}.txt')
            if not eid_file.exists(): continue
            
            sample_eids = np.loadtxt(eid_file, dtype=int)
            df = df_full[df_full.index.isin(sample_eids)]
            actual_n = len(df)
            
            print(f'\n[{target_name} | {data_name} | n={sample_size} ({actual_n} rows)]')
            start_time = time.time()
            metrics = gnn_analysis(df, data_name, target_name, sample_size)
            elapsed = time.time() - start_time
            print(f'  Time: {elapsed:.2f}s')

            results_file = results_dir / f'gnn_scaling_law_results_{target_name}.csv'
            row_df = pd.DataFrame([{'target_name': target_name, 'data_name': data_name, 'sample_size': sample_size, 'actual_n': actual_n, **metrics, 'elapsed_time_sec': elapsed}])
            row_df.to_csv(results_file, mode='a', header=not results_file.exists(), index=False)